# 00 · Quick look at the data
**Phase 1 (Ask).** Pull a few headline numbers so the project brief starts from real facts.
No cleaning yet. That happens in Phase 3 (Process).

In [ ]:
from completejourney_py import get_data

# Load only what we need now. promotions (~21M rows) is skipped until Prepare.
data = get_data(["transactions", "campaigns", "campaign_descriptions", "demographics"])
tx, camps, camp_desc, demo = (data[k] for k in ["transactions", "campaigns", "campaign_descriptions", "demographics"])
tx.head()

## Sales tree
Sales = households × trips per household × basket value. The last line checks that the tree multiplies out exactly.

In [ ]:
sales        = tx["sales_value"].sum()
households   = tx["household_id"].nunique()
trips        = tx["basket_id"].nunique()
trips_per_hh = trips / households
basket_value = sales / trips

print(f"Transaction lines   : {len(tx):,}")
print(f"Date range          : {tx['transaction_timestamp'].min():%d %b %Y} to {tx['transaction_timestamp'].max():%d %b %Y}")
print(f"Stores              : {tx['store_id'].nunique():,}")
print(f"Total sales         : ${sales:,.0f}")
print(f"Households          : {households:,}")
print(f"Trips               : {trips:,}")
print(f"Trips per household : {trips_per_hh:.1f}")
print(f"Basket value        : ${basket_value:.2f}")
print(f"Check (H x T x B)   : ${households * trips_per_hh * basket_value:,.0f}")

## Discounts
Older releases of this data store discounts as negative numbers; this package stores them as positives. `abs()` gives the right answer either way. Before adding up a column, check which way round its numbers are stored.

In [ ]:
retail_disc = tx["retail_disc"].abs().sum()
coupon_disc = tx["coupon_disc"].abs().sum()

print(f"Retail discount     : ${retail_disc:,.0f}")
print(f"Coupon discount     : ${coupon_disc:,.0f}")
print(f"Discount / sales    : {(retail_disc + coupon_disc) / sales:.1%}")
print(f"Lines with discount : {((tx['retail_disc'] != 0) | (tx['coupon_disc'] != 0)).mean():.0%}")

## Campaigns and demographics coverage

In [ ]:
reached = camps["household_id"].nunique()
print(f"Campaigns           : {camp_desc['campaign_id'].nunique()}")
print(camp_desc["campaign_type"].value_counts().sort_index().to_string())
print(f"Households reached  : {reached:,} of {households:,} ({reached / households:.0%})")
print(f"Households with demographics: {demo['household_id'].nunique():,} of {households:,}")